# BAT Construct Sampling

Draw a stratified sample of **60 posts (15 per BAT construct)** from
`bat_score_pos.csv`, excluding any `post_id` already present in
`merged_post40_llm.csv`.

**Schema note:** there is no single "category" column. Instead each row has
4 binary YES/NO columns — `EX`, `EMO`, `COG`, `MD` — one per BAT construct.
This notebook treats those 4 columns as the "4 categories" and samples 15
posts per construct where that construct is `YES`.

**Assumptions (change the Config cell if these don't match what you want):**
- Only `row_type == 'post'` rows are eligible (not comment-level rows).
- Sampling is done **independently per construct**. A post that is `YES` on
  more than one construct could therefore be selected under multiple
  constructs — it is *not* forced to be mutually exclusive across the 4
  samples. Set `MUTUALLY_EXCLUSIVE = True` in Config if you'd rather each
  post only ever appear once across the full 60.
- Exclusion against `merged_post40_llm.csv` is by `post_id`.

Built for **large files**: `merged_post40_llm.csv` is read fully (it's the
smaller "already used" file) to build an exclusion set, but
`bat_score_pos.csv` is **streamed in chunks** and sampled with
**reservoir sampling**, so memory stays flat no matter how big the file is.

**Run order:** run all cells top to bottom. Adjust the Config cell first if needed.


In [1]:
import random
from pathlib import Path

import pandas as pd


## Config

In [2]:
# ============================== CONFIG ===================================

BAT_SCORE_POS_CSV = "bat_score_pos.csv"     # the big file we sample FROM
MERGED_LLM_CSV = "merged_post40_llm.csv"    # the file whose posts we EXCLUDE
OUTPUT_CSV = "bat_sample_60.csv"

ID_COL = "post_id"                          # shared id column in both files
ROW_TYPE_COL = "row_type"
ROW_TYPE_VALUE = "post"                     # only sample post-level rows

CATEGORIES = ["EX", "EMO", "COG", "MD"]     # the 4 BAT construct columns
POSITIVE_VALUE = "YES"
SAMPLES_PER_CATEGORY = 15

# If True: once a post_id is sampled for one construct, it is excluded from
# the reservoirs of the remaining constructs (each post appears at most once
# across the whole 60-row output). If False (default): sampling per
# construct is fully independent.
MUTUALLY_EXCLUSIVE = False

CHUNKSIZE = 100_000       # rows per chunk while streaming the big file
RANDOM_STATE = 42         # project convention: reproducible sampling
# ===========================================================================


## Step 1 — Inspect headers (sanity check)

In [3]:
def inspect_headers(path, n_preview=2):
    p = Path(path)
    if not p.exists():
        print(f"[MISSING] {path} not found in current directory.")
        return []
    df_head = pd.read_csv(path, nrows=n_preview, dtype=str)
    print(f"=== {path} ===")
    print(f"Columns ({len(df_head.columns)}): {list(df_head.columns)}")
    return list(df_head.columns)


bat_header = inspect_headers(BAT_SCORE_POS_CSV)
merged_header = inspect_headers(MERGED_LLM_CSV)

missing = [c for c in [ID_COL, ROW_TYPE_COL] + CATEGORIES if c not in bat_header]
if missing:
    raise SystemExit(f"bat_score_pos.csv is missing expected columns: {missing}")
if ID_COL not in merged_header:
    raise SystemExit(f"merged_post40_llm.csv is missing expected id column: {ID_COL}")
print("Header check OK.")


=== bat_score_pos.csv ===
Columns (16): ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning']
=== merged_post40_llm.csv ===
Columns (17): ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'batch']
Header check OK.


## Step 2 — Build exclusion set from `merged_post40_llm.csv`

In [4]:
def build_exclusion_set(path, id_col):
    print(f"Building exclusion set from {path} using id column '{id_col}'...")
    df = pd.read_csv(path, usecols=[id_col], dtype=str)
    ids = set(df[id_col].dropna().astype(str).str.strip())
    print(f"  -> {len(ids)} unique post_ids to exclude")
    return ids


exclude_ids = build_exclusion_set(MERGED_LLM_CSV, ID_COL)


Building exclusion set from merged_post40_llm.csv using id column 'post_id'...
  -> 40 unique post_ids to exclude


## Step 3 — Stratified reservoir sample from `bat_score_pos.csv`\n\nStreams the file in chunks; each eligible post gets an equal chance of being picked per construct. Memory stays flat regardless of file size.

In [5]:
def stratified_reservoir_sample(
    path, id_col, row_type_col, row_type_value, categories, positive_value,
    per_category, exclude_ids, chunksize, random_state, mutually_exclusive,
):
    rng = random.Random(random_state)

    reservoirs = {cat: [] for cat in categories}      # cat -> list[Series]
    reservoir_ids = {cat: [] for cat in categories}    # cat -> list[post_id], parallel to reservoirs
    seen_counts = {cat: 0 for cat in categories}
    picked_ids_global = set()  # used only when mutually_exclusive=True

    total_rows = 0
    for chunk in pd.read_csv(
        path, chunksize=chunksize, dtype=str, keep_default_na=False
    ):
        total_rows += len(chunk)

        chunk = chunk[chunk[row_type_col] == row_type_value]
        chunk["_norm_id"] = chunk[id_col].astype(str).str.strip()
        chunk = chunk[~chunk["_norm_id"].isin(exclude_ids)]

        for cat in categories:
            cat_rows = chunk[chunk[cat].astype(str).str.strip() == positive_value]
            for _, row in cat_rows.iterrows():
                pid = row["_norm_id"]

                if mutually_exclusive and pid in picked_ids_global:
                    continue

                seen_counts[cat] += 1
                res = reservoirs[cat]
                res_ids = reservoir_ids[cat]

                if len(res) < per_category:
                    res.append(row.drop(labels=["_norm_id"]))
                    res_ids.append(pid)
                    if mutually_exclusive:
                        picked_ids_global.add(pid)
                else:
                    j = rng.randint(0, seen_counts[cat] - 1)
                    if j < per_category:
                        if mutually_exclusive:
                            picked_ids_global.discard(res_ids[j])
                            picked_ids_global.add(pid)
                        res[j] = row.drop(labels=["_norm_id"])
                        res_ids[j] = pid

        sizes = ", ".join(f"{c}:{len(r)}" for c, r in reservoirs.items())
        print(f"  processed {total_rows:,} rows so far... (reservoir sizes: {{{sizes}}})")

    for cat in categories:
        got = len(reservoirs[cat])
        if got < per_category:
            print(f"[WARNING] construct '{cat}' only had {got} eligible posts (wanted {per_category}).")

    frames = []
    for cat in categories:
        if reservoirs[cat]:
            df_cat = pd.DataFrame(reservoirs[cat])
            df_cat["sampled_for_construct"] = cat
            frames.append(df_cat)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


result = stratified_reservoir_sample(
    path=BAT_SCORE_POS_CSV,
    id_col=ID_COL,
    row_type_col=ROW_TYPE_COL,
    row_type_value=ROW_TYPE_VALUE,
    categories=CATEGORIES,
    positive_value=POSITIVE_VALUE,
    per_category=SAMPLES_PER_CATEGORY,
    exclude_ids=exclude_ids,
    chunksize=CHUNKSIZE,
    random_state=RANDOM_STATE,
    mutually_exclusive=MUTUALLY_EXCLUSIVE,
)

result.shape


  processed 12,237 rows so far... (reservoir sizes: {EX:15, EMO:15, COG:15, MD:15})


(60, 17)

## Step 4 — Save output

In [6]:
if result.empty:
    raise SystemExit("No rows sampled -- check Config values.")

result.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(result)} sampled rows -> {OUTPUT_CSV}")
result["sampled_for_construct"].value_counts()


Saved 60 sampled rows -> bat_sample_60.csv


sampled_for_construct
EX     15
EMO    15
COG    15
MD     15
Name: count, dtype: int64